In [ ]:
# notebooks/04_shot_map.ipynb  セル1
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from mplsoccer import VerticalPitch

# src フォルダを import の探索先に追加する(自作の loader.py などを使うため)
sys.path.append("../src")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# 図の色(フェーズ3と同じ暗い緑の背景)
BG_COLOR = "#22312b"      # 背景・ピッチの色
LINE_COLOR = "#c7d5cc"    # ピッチのラインの色
TEXT_COLOR = "#e8efe9"    # 文字の色
GOAL_COLOR = "#f2c14e"    # ゴールになったシュートの色(金色)
MISS_COLOR = "#6cb2eb"    # ゴールにならなかったシュートの色(青)

# マーカーの大きさ = SIZE_BASE + SIZE_SCALE × xG
SIZE_BASE = 80
SIZE_SCALE = 1000


In [ ]:
# notebooks/04_shot_map.ipynb  セル2
from pitchviz.loader import get_events

TEAM_NAME = "Japan"       # フェーズ2のメモの値に置き換える(例)
MATCH_ID = 7584           # フェーズ2のメモの値に置き換える(例)
MATCH_DATE = "2018-07-02" # フェーズ2のメモの日付に置き換える(例)

events = get_events(MATCH_ID)
teams = list(events["team"].unique())
opponent_name = [t for t in teams if t != TEAM_NAME][0]

print(events.shape)
print(teams, "->", TEAM_NAME, "vs", opponent_name)


In [ ]:
# notebooks/04_shot_map.ipynb  セル3
shots = events[(events["type"] == "Shot") & (events["team"] == TEAM_NAME)].copy()
print(shots["period"].value_counts().sort_index())   # どの時間帯のシュートか(5 は PK戦)

shots = shots[shots["period"] < 5].copy()            # PK戦(period が 5)のシュートを除く

shots["x"] = shots["location"].str[0]
shots["y"] = shots["location"].str[1]
shots["xg"] = shots["shot_statsbomb_xg"].fillna(0.0)
shots["is_goal"] = shots["shot_outcome"] == "Goal"

print("shots:", len(shots), " goals:", int(shots["is_goal"].sum()), " xG total:", round(shots["xg"].sum(), 2))
print(shots["shot_outcome"].value_counts())
shots[["minute", "period", "player", "x", "y", "xg", "shot_outcome", "shot_type", "is_goal"]].head()


In [ ]:
# notebooks/04_shot_map.ipynb  セル4
pitch = VerticalPitch(pitch_type="statsbomb", half=True, pad_bottom=0.5, goal_type="box",
                      pitch_color=BG_COLOR, line_color=LINE_COLOR)
fig, ax = pitch.draw(figsize=(8, 7))
fig.set_facecolor(BG_COLOR)

first = shots.head(1)   # 先頭の1行(1本のシュート)
pitch.scatter(first.x, first.y, s=300, c=MISS_COLOR, edgecolors=LINE_COLOR, ax=ax)
plt.show()


In [ ]:
# notebooks/04_shot_map.ipynb  セル5
pitch = VerticalPitch(pitch_type="statsbomb", half=True, pad_bottom=0.5, goal_type="box",
                      pitch_color=BG_COLOR, line_color=LINE_COLOR)
fig, ax = pitch.draw(figsize=(8, 7))
fig.set_facecolor(BG_COLOR)

pitch.scatter(shots.x, shots.y, s=300, c=MISS_COLOR, edgecolors=LINE_COLOR, alpha=0.8, ax=ax)
plt.show()


In [ ]:
# notebooks/04_shot_map.ipynb  セル6(課題1)
shots["marker_size"] = SIZE_BASE + SIZE_SCALE * shots["xg"]

print(shots[["xg", "size"]].round(2).sort_values("xg").head(3))
print(shots[["xg", "size"]].round(2).sort_values("xg").tail(3))


In [ ]:
# notebooks/04_shot_map.ipynb  セル7(課題2)
pitch = VerticalPitch(pitch_type="statsbomb", half=True, pad_bottom=0.5, goal_type="box",
                      pitch_color=BG_COLOR, line_color=LINE_COLOR)
fig, ax = pitch.draw(figsize=(8, 7))
fig.set_facecolor(BG_COLOR)

goals = shots[shots["is_goal"]]        # ゴールになったシュート
no_goals = shots[~shots["is_goal"]]    # ゴールにならなかったシュート

pitch.scatter(no_goals.x, no_goals.y, s=no_goals.marker_size, c=MISS_COLOR, edgecolors=LINE_COLOR,
              alpha=0.55, linewidths=1.2, ax=ax)
pitch.scatter(goals.x, goals.y, s=goals.marker_size, c=GOAL_COLOR, edgecolors="white",
              alpha=1.0, linewidths=2, ax=ax)
plt.show()

print(len(goals) + len(no_goals) == len(shots))


In [ ]:
# notebooks/04_shot_map.ipynb  セル8
pitch = VerticalPitch(pitch_type="statsbomb", half=True, pad_bottom=0.5, goal_type="box",
                      pitch_color=BG_COLOR, line_color=LINE_COLOR)
fig, axs = pitch.grid(figheight=9, title_height=0.10, grid_height=0.78, endnote_height=0.07,
                      title_space=0, endnote_space=0, axis=False)
fig.set_facecolor(BG_COLOR)
ax = axs["pitch"]

goals = shots[shots["is_goal"]]
no_goals = shots[~shots["is_goal"]]

pitch.scatter(no_goals.x, no_goals.y, s=no_goals["marker_size"], c=MISS_COLOR, edgecolors=LINE_COLOR,
              alpha=0.55, linewidths=1.2, ax=ax)
pitch.scatter(goals.x, goals.y, s=goals["marker_size"], c=GOAL_COLOR, edgecolors="white",
              alpha=1.0, linewidths=2, ax=ax)

# 凡例用の見本(データを描かない、空の点)
handles = [
    ax.scatter([], [], s=250, c=GOAL_COLOR, edgecolors="white", linewidths=2,
               label=f"Goal ({len(goals)})"),
    ax.scatter([], [], s=250, c=MISS_COLOR, edgecolors=LINE_COLOR, alpha=0.55, linewidths=1.2,
               label=f"No goal ({len(no_goals)})"),
]
for sample_xg in (0.05, 0.2, 0.5):
    handles.append(ax.scatter([], [], s=SIZE_BASE + SIZE_SCALE * sample_xg, c="none",
                              edgecolors=LINE_COLOR, linewidths=1.2, label=f"xG {sample_xg}"))
ax.legend(handles=handles, loc="lower left", facecolor=BG_COLOR, edgecolor=LINE_COLOR,
          labelcolor=TEXT_COLOR, labelspacing=1.6, borderpad=1.0, fontsize=11)

total_xg = shots["xg"].sum()
axs["title"].text(0.5, 0.68, f"{TEAM_NAME} shot map vs {opponent_name}",
                  color=TEXT_COLOR, ha="center", va="center", fontsize=24,
                  transform=axs["title"].transAxes)
axs["title"].text(0.5, 0.18,
                  f"{MATCH_DATE}   |   {len(shots)} shots, {len(goals)} goals, xG {total_xg:.2f}",
                  color=LINE_COLOR, ha="center", va="center", fontsize=13,
                  transform=axs["title"].transAxes)
axs["endnote"].text(0.01, 0.5, "Data: StatsBomb Open Data   |   marker size = xG",
                    color=TEXT_COLOR, ha="left", va="center", fontsize=11,
                    transform=axs["endnote"].transAxes)
plt.show()


In [ ]:
# notebooks/04_shot_map.ipynb  セル9
from pitchviz.pass_map import add_logo, save_figure

LOGO_PATH = "../assets/statsbomb_logo_white.png"
OUTPUT_DIR = Path("../outputs")

add_logo(fig, axs["endnote"], LOGO_PATH)
out_path = save_figure(fig, OUTPUT_DIR / "shot_map_japan_belgium.png")   # 例。自分の試合に合わせて変える
print("saved:", out_path)
fig


In [ ]:
# notebooks/04_shot_map.ipynb  セル10
from pitchviz.loader import get_shots
from pitchviz.pass_map import make_filename
from pitchviz.shot_map import draw_shot_map, summarize_shots

print(make_filename("Japan", "Belgium", prefix="shot_map"))
print(make_filename("Japan Women's", "Norway Women's", prefix="shot_map"))

checked = get_shots(events, TEAM_NAME)
print(len(checked) == len(shots))
print(summarize_shots(checked))


In [ ]:
# notebooks/04_shot_map.ipynb  セル11
LOGO_PATH = "../assets/statsbomb_logo_white.png"
OUTPUT_DIR = Path("../outputs")

for team in teams:
    team_shots = get_shots(events, team)
    opponent = [t for t in teams if t != team][0]

    fig = draw_shot_map(team_shots, team, opponent, MATCH_DATE, logo_path=LOGO_PATH)
    out_path = save_figure(fig, OUTPUT_DIR / make_filename(team, opponent, prefix="shot_map"))
    print("saved:", out_path)
